In [1]:
!pip install requests pandas tqdm


In [4]:
import requests
import pandas as pd
from tqdm import tqdm
import time

BASE = "https://open-bus-stride-api.hasadna.org.il"

def fetch_with_retry(url, params, max_retries=5, timeout=120):
    """שליפה עם retry אוטומטי במקרה של timeout"""
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except (requests.ReadTimeout, requests.ConnectionError) as e:
            wait = (attempt + 1) * 5
            print(f"  ⚠️  ניסיון {attempt+1}/{max_retries} נכשל — ממתין {wait}ש'...")
            time.sleep(wait)
        except Exception as e:
            print(f"  ❌ שגיאה: {e}")
            raise
    raise Exception(f"נכשל אחרי {max_retries} ניסיונות")

def fetch_all_rides():
    rides = []
    offset = 0
    params = {
        "limit": 100,
        "scheduled_start_time_from": "2024-01-05T00:00:00Z",
        "scheduled_start_time_to":   "2024-01-11T23:59:59Z",
        "gtfs_route__route_long_name_contains": "תל אביב",
    }

    # קבל כמות כוללת
    count_params = {**params, "limit": 1, "get_count": "true"}
    total = fetch_with_retry(f"{BASE}/siri_rides/list", count_params)
    print(f"סה\"כ {total} נסיעות לשליפה")

    pbar = tqdm(total=total, unit="נסיעות")

    while True:
        params["offset"] = offset
        batch = fetch_with_retry(f"{BASE}/siri_rides/list", params)

        if not isinstance(batch, list) or not batch:
            break

        rides.extend(batch)
        pbar.update(len(batch))

        if len(batch) < 100:
            break

        offset += len(batch)
        time.sleep(0.15)  # עיכוב קל בין בקשות

    pbar.close()
    print(f"\n✅ סה\"כ {len(rides)} נסיעות")
    return rides

rides = fetch_all_rides()

סה"כ 107383 נסיעות לשליפה




  0%|                                                                                   | 0/107383 [00:00<?, ?נסיעות/s]

  0%|                                                                       | 100/107383 [00:22<6:41:36,  4.45נסיעות/s]

  0%|▏                                                                      | 200/107383 [00:35<5:06:44,  5.82נסיעות/s]

  0%|▏                                                                      | 300/107383 [00:47<4:24:39,  6.74נסיעות/s]

  0%|▎                                                                      | 400/107383 [01:01<4:14:00,  7.02נסיעות/s]

  0%|▎                                                                      | 500/107383 [01:12<3:56:55,  7.52נסיעות/s]

  1%|▍                                                                      | 600/107383 [01:27<4:05:17,  7.26נסיעות/s]

  1%|▍                                                                      | 700/107383 [01:40<4:01:58,  7.35נסיעות/s]

  1%|▌                        

  ⚠️  ניסיון 1/5 נכשל — ממתין 5ש'...




 12%|████████▌                                                              | 13000/107383 [20:03<35:08, 44.76נסיעות/s]

 12%|████████▎                                                           | 13100/107383 [20:05<20:38:51,  1.27נסיעות/s]

 12%|████████▎                                                           | 13200/107383 [20:08<14:40:00,  1.78נסיעות/s]

 12%|████████▍                                                           | 13300/107383 [20:11<10:27:28,  2.50נסיעות/s]

 12%|████████▌                                                            | 13400/107383 [20:13<7:30:27,  3.48נסיעות/s]

 13%|████████▋                                                            | 13500/107383 [20:16<5:27:22,  4.78נסיעות/s]

 13%|████████▋                                                            | 13600/107383 [20:19<4:02:19,  6.45נסיעות/s]

 13%|████████▊                                                            | 13700/107383 [20:21<3:00:41,  8.64נסיעות/s]

 13%|████████▊                


✅ סה"כ 107383 נסיעות


In [5]:
def fetch_stops_for_ride(gtfs_ride_id):
    r = requests.get(f"{BASE}/gtfs_ride_stops/list", params={
        "limit": 200,
        "gtfs_ride_ids": gtfs_ride_id,
        "order_by": "stop_sequence"
    }, timeout=30)
    return r.json() if r.ok else []

def haversine(lat1, lon1, lat2, lon2):
    from math import radians, sin, cos, sqrt, atan2
    R = 6371
    dlat = radians(lat2-lat1); dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# בנה cache — נסיעה אחת לכל route ייחודי
route_cache = {}
seen = {}
for ride in rides:
    rid = ride.get("gtfs_ride__gtfs_route_id")
    if rid and rid not in seen and ride.get("gtfs_ride_id"):
        seen[rid] = ride["gtfs_ride_id"]

print(f"שולף תחנות ל-{len(seen)} routes ייחודיים...")
for route_id, gtfs_ride_id in tqdm(seen.items()):
    try:
        stops = fetch_stops_for_ride(gtfs_ride_id)
        if not isinstance(stops, list): stops = []
        dist = 0
        for i in range(1, len(stops)):
            a, b = stops[i-1], stops[i]
            if a.get("gtfs_stop__lat") and b.get("gtfs_stop__lat"):
                dist += haversine(a["gtfs_stop__lat"], a["gtfs_stop__lon"],
                                   b["gtfs_stop__lat"], b["gtfs_stop__lon"])
        route_cache[route_id] = {
            "from_city":  stops[0].get("gtfs_stop__city","")  if stops else "",
            "from_stop":  stops[0].get("gtfs_stop__name","")  if stops else "",
            "to_city":    stops[-1].get("gtfs_stop__city","") if stops else "",
            "to_stop":    stops[-1].get("gtfs_stop__name","") if stops else "",
            "stop_count": len(stops),
            "dist_km":    round(dist, 3),
        }
    except Exception as e:
        route_cache[route_id] = {}
    time.sleep(0.15)

print(f"✅ cache מוכן: {len(route_cache)} routes")

שולף תחנות ל-3623 routes ייחודיים...




  0%|                                                                                         | 0/3623 [00:00<?, ?it/s]

  0%|                                                                               | 1/3623 [00:01<1:22:22,  1.36s/it]

  0%|                                                                               | 2/3623 [00:02<1:17:46,  1.29s/it]

  0%|                                                                               | 3/3623 [00:03<1:15:59,  1.26s/it]

  0%|                                                                               | 4/3623 [00:05<1:16:36,  1.27s/it]

  0%|                                                                               | 5/3623 [00:06<1:21:46,  1.36s/it]

  0%|▏                                                                              | 6/3623 [00:07<1:20:04,  1.33s/it]

  0%|▏                                                                              | 7/3623 [00:09<1:19:33,  1.32s/it]

  0%|▏                        

✅ cache מוכן: 3623 routes


In [6]:
from datetime import datetime, timezone

def fmt_time(s):
    if not s: return ""
    return datetime.fromisoformat(s).strftime("%H:%M:%S")

def fmt_date(s):
    if not s: return ""
    return datetime.fromisoformat(s).strftime("%d/%m/%Y")

def day_he(s):
    if not s: return ""
    days = ["שני","שלישי","רביעי","חמישי","שישי","שבת","ראשון"]
    return days[datetime.fromisoformat(s).weekday()]

def round_hour(s):
    if not s: return ""
    return f"{datetime.fromisoformat(s).hour:02d}:00"

def dur_min(a, b):
    if not a or not b: return ""
    m = round((datetime.fromisoformat(b) - datetime.fromisoformat(a)).total_seconds() / 60)
    return m if 0 < m < 600 else ""

rows = []
for ride in rides:
    dep  = ride.get("gtfs_ride__start_time") or ride.get("scheduled_start_time","")
    arr  = ride.get("gtfs_ride__end_time","")
    plan = dur_min(dep, arr)
    act  = ride.get("duration_minutes") if ride.get("duration_minutes",0) > 0 else ""
    diff = (act - plan) if isinstance(plan,int) and isinstance(act,int) else ""
    s    = route_cache.get(ride.get("gtfs_ride__gtfs_route_id"), {})
    dist = s.get("dist_km", 0)
    sp_plan = round(dist / (plan/60), 1) if dist and isinstance(plan,int) and plan > 0 else ""
    sp_act  = round(dist / (act/60),  1) if dist and isinstance(act,int)  and act  > 0 else ""

    rows.append({
        "תאריך":                   fmt_date(dep),
        "יום":                     day_he(dep),
        "שעה (עגולה)":            round_hour(dep),
        "מספר קו":                 ride.get("gtfs_route__route_short_name",""),
        "שם הקו":                  ride.get("gtfs_route__route_long_name",""),
        "כיוון":                   ride.get("gtfs_route__route_direction",""),
        "אלטרנטיבה":               ride.get("gtfs_route__route_alternative",""),
        "חברה (agency_name)":      ride.get("gtfs_route__agency_name",""),
        "סוג מסלול (route_type)":  "אוטובוס" if ride.get("gtfs_route__route_type")=="3" else ride.get("gtfs_route__route_type",""),
        "עיר מוצא":                s.get("from_city",""),
        "תחנת מוצא":               s.get("from_stop",""),
        "עיר יעד":                 s.get("to_city",""),
        "תחנת יעד":                s.get("to_stop",""),
        "כמות תחנות":              s.get("stop_count",""),
        "אורך מסלול (קמ)":         dist or "",
        "זמן יציאה מתוכנן":        fmt_time(dep),
        "זמן הגעה מתוכנן":         fmt_time(arr),
        "משך מתוכנן (דק)":         plan,
        "משך בפועל (דק)":          act,
        "הפרש (דק)":               diff,
        "מהירות מתוכננת (קמש)":    sp_plan,
        "מהירות בפועל (קמש)":      sp_act,
        "מזהה SIRI":               ride.get("id",""),
    })

df = pd.DataFrame(rows)
df.to_csv("telaviv_buses_jan2024.csv", index=False, encoding="utf-8-sig")
print(f"✅ נשמר: telaviv_buses_jan2024.csv")
print(df.shape)
df.head(3)

  0%|                                                                                    | 0/107383 [16:12:27<?, ?it/s]


✅ נשמר: telaviv_buses_jan2024.csv
(107383, 23)


,תאריך,יום,שעה (עגולה),מספר קו,שם הקו,כיוון,אלטרנטיבה,חברה (agency_name),סוג מסלול (route_type),עיר מוצא,...,כמות תחנות,אורך מסלול (קמ),זמן יציאה מתוכנן,זמן הגעה מתוכנן,משך מתוכנן (דק),משך בפועל (דק),הפרש (דק),מהירות מתוכננת (קמש),מהירות בפועל (קמש),מזהה SIRI
0,05/01/2024,שישי,00:00,61,הכובשים/דניאל-תל אביב יפו<->מסוף עמידר-רמת גן-10,1,0,דן,אוטובוס,תל אביב יפו,...,38,11.718,00:00:00,00:39:26,39,46,7,18.0,15.3,58601679
1,05/01/2024,שישי,00:00,2,פרופ' ישעיהו ליבוביץ'/הרב יעקב טראב-תל אביב יפ...,2,0,דן,אוטובוס,תל אביב יפו,...,51,17.727,00:00:00,00:58:59,59,54,-5,18.0,19.7,58601680
2,05/01/2024,שישי,00:00,699,הרכבת/ראש פינה-תל אביב יפו<->ת. מרכזית נתניה-נ...,2,#,מטרופולין,אוטובוס,תל אביב יפו,...,42,31.934,00:00:00,01:03:59,64,64,0,29.9,29.9,58601681
